In [5]:
import threading
import time
def download_file(file_id:int,delay:int):
    time.sleep(delay)
print("=== 串行下载 ===")
start = time.time()
download_file(1,2)
download_file(2,1)
download_file(3,3)
print(time.time()-start)
print("=== 多线程下载 ===")
t1 = threading.Thread(target=download_file,args=(1,2))
t2 = threading.Thread(target=download_file,args=(2,1))
t3 = threading.Thread(target=download_file,args=(3,3))
t1.start()
t2.start()
t3.start()
t1.join()
t2.join()
t3.join()
print(time.time()-start)

=== 串行下载 ===
6.001379489898682
=== 多线程下载 ===
9.0030357837677


In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor,as_completed
def fetch_page(url:str):
    time.sleep(1)
urls = [f"https:/example.com/page/{i}" for i in range(10)]
start = time.time()
with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(fetch_page,url):url for url in urls}
    for future in futures:
        url = futures[future]
        result = future.result()
elapsed = time.time() - start
print(elapsed)

2.003220796585083


In [23]:
import threading
counter = 0
lock = threading.Lock()
def safe_increment():
    global counter
    for _ in range(100000):
        with lock:
            current = counter
            counter += 1
threads = [threading.Thread(target=safe_increment)for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()
print(counter)

1000000


In [28]:
import time
import random
import threading
from concurrent.futures import ThreadPoolExecutor,as_completed
class TokenCounter:
    def __init__(self):
        self._lock = threading.Lock()
        self.total_tokens = 0
        self.success = 0
        self.failed = 0
    def record_success(self,tokens:int):
        with self._lock:
            self.total_tokens += tokens
            self.success += 1
    def record_failure(self):
        with self._lock:
            self.failed += 1
def call_llm(prompt:str,counter:TokenCounter) -> str:
    time.sleep(random.uniform(0.5,2.0))
    if random.random() < 0.1:
        counter.record_failure()
        raise ConnectionError('API 500 Error')
    tokens = random.randint(100,500)
    counter.record_success(tokens)
    return f"回复: {prompt[:20]}... (消耗 {tokens} tokens)"            
def main():
    prompts = [f"请解释 {topic}" for topic in 
               ["RAG", "Agent", "Transformer", "LoRA", "RLHF", 
                "Fine-tuning", "Prompt Engineering", "Embedding", 
                "Vector DB", "Chain of Thought"]]
    counter = TokenCounter()
    start = time.time()
    with ThreadPoolExecutor(max_workers=3) as pool:
        futures={
            pool.submit(call_llm,prompt,counter):prompt
            for prompt in prompts
        }
        for future in as_completed(futures):
            prompt = futures[future]
            try:
                result = future.result(timeout=10)
            except Exception as e:
                print(f"  ❌ {prompt[:30]}... → {e}")
        elapsed = time.time() - start
        print('success',counter.success)
        print('failed',counter.failed)
        print('total_tokens',counter.total_tokens)
        print('time',elapsed)
if __name__ == "__main__":
    main()    

success 10
failed 0
total_tokens 2632
time 4.824587106704712
